# 🛣️ SafeRoad AI — YOLOv8 Road Hazard Detector
### Google Colab Training Notebook

**Detects:**
- 🕳️ Potholes
- 🕸️ Alligator Cracks
- ➖ Longitudinal Cracks
- 🪨 Severe Edge Erosion

---
### ✅ Before Running:
1. Go to **Runtime → Change runtime type → T4 GPU** (Free) or A100 (Colab Pro)
2. Run cells **one by one** from top to bottom
3. At the end, download your trained `best.pt` model

---

## ⚙️ Step 1 — Check GPU

In [ ]:
# Check GPU availability
!nvidia-smi

import torch
print(f"\n✅ PyTorch version: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 📦 Step 2 — Install Dependencies

In [ ]:
!pip install ultralytics roboflow kaggle -q
print("✅ All packages installed!")

## 📂 Step 3 — Load Dataset

**Choose ONE option below (A, B, or C). Skip the others.**

### 🅰️ Option A — Kaggle Dataset (Paste your Kaggle dataset link)

In [ ]:
# ─────────────────────────────────────────────────────
# OPTION A: Download from Kaggle
# ─────────────────────────────────────────────────────
# Step 1: Upload your kaggle.json API key
from google.colab import files
print("Upload your kaggle.json file (from kaggle.com → Account → Create New Token)")
uploaded = files.upload()  # Upload kaggle.json here

import os
os.makedirs('/root/.kaggle', exist_ok=True)
os.rename('/content/kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print("✅ Kaggle API key configured!")

In [ ]:
# ─────────────────────────────────────────────────────
# Paste your Kaggle dataset URL here:
# Example: https://www.kaggle.com/datasets/atulyakumar98/pothole-detection-dataset
#          → dataset_id = "atulyakumar98/pothole-detection-dataset"
# ─────────────────────────────────────────────────────

KAGGLE_DATASET_ID = "atulyakumar98/pothole-detection-dataset"  # ← CHANGE THIS

!kaggle datasets download -d {KAGGLE_DATASET_ID} --unzip -p /content/dataset
!ls /content/dataset
print("✅ Kaggle dataset downloaded!")

### 🅱️ Option B — Roboflow Dataset (Free, No API key needed)

In [ ]:
# ─────────────────────────────────────────────────────
# OPTION B: Download from Roboflow
# Go to https://universe.roboflow.com → find a pothole dataset
# → Click "Download" → Choose YOLOv8 format → Copy the code snippet below
# ─────────────────────────────────────────────────────

from roboflow import Roboflow

# Replace with your Roboflow API key and dataset details:
rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")  # ← CHANGE THIS
project = rf.workspace("WORKSPACE").project("PROJECT_NAME")  # ← CHANGE THIS
dataset = project.version(1).download("yolov8")

DATA_YAML = dataset.location + "/data.yaml"
print(f"✅ Dataset ready at: {DATA_YAML}")

### 🅲 Option C — Synthetic Dataset (No download needed — works offline)

In [ ]:
# ─────────────────────────────────────────────────────
# OPTION C: Generate Synthetic Road Dataset
# No internet / Kaggle account needed!
# ─────────────────────────────────────────────────────

import numpy as np
import cv2
import random
from pathlib import Path

DATASET_DIR = Path("/content/synthetic_road")
IMG_SIZE = 640
NUM_TRAIN = 300   # ← Increase for better accuracy (try 500)
NUM_VAL   = 80

for split in ["train", "val"]:
    (DATASET_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (DATASET_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

def make_asphalt(w, h):
    base = random.randint(60, 110)
    img = np.full((h, w, 3), base, dtype=np.uint8)
    noise = np.random.normal(0, random.uniform(8, 18), (h, w, 3)).astype(np.int16)
    img = np.clip(img.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    if random.random() < 0.6:
        lx = random.randint(int(w*0.1), int(w*0.9))
        cv2.line(img, (lx,0), (lx+random.randint(-30,30),h), (220,220,220), random.randint(6,14))
    grad = np.linspace(0.75, 1.15, h)[:,None,None]
    return np.clip(img * grad, 0, 255).astype(np.uint8)

def draw_pothole(img, labels):
    h,w = img.shape[:2]
    cx,cy = random.randint(int(w*0.2),int(w*0.8)), random.randint(int(h*0.25),int(h*0.8))
    rx,ry = random.randint(int(w*0.05),int(w*0.15)), 0
    ry = int(rx*random.uniform(0.45,0.85))
    ang = random.randint(0,180)
    cv2.ellipse(img,(cx,cy),(rx,ry),ang,0,360,(20,20,20),-1)
    cv2.ellipse(img,(cx,cy),(int(rx*.75),int(ry*.75)),ang,0,360,(10,10,10),-1)
    bx1,by1,bx2,by2 = max(0,cx-rx-4),max(0,cy-ry-4),min(w,cx+rx+4),min(h,cy+ry+4)
    labels.append(f"0 {((bx1+bx2)/2)/w:.6f} {((by1+by2)/2)/h:.6f} {(bx2-bx1)/w:.6f} {(by2-by1)/h:.6f}")

def draw_crack(img, labels):
    h,w = img.shape[:2]
    cx,cy = random.randint(int(w*0.25),int(w*0.75)), random.randint(int(h*0.25),int(h*0.75))
    rx,ry = random.randint(int(w*0.08),int(w*0.18)), 0
    ry = int(rx*random.uniform(0.6,1.0))
    pts = [(int(cx+random.gauss(0,rx*.45)), int(cy+random.gauss(0,ry*.45))) for _ in range(random.randint(14,24))]
    pts = [(np.clip(p[0],10,w-10), np.clip(p[1],10,h-10)) for p in pts]
    for i in range(len(pts)):
        for j in range(i+1, min(i+4,len(pts))):
            if random.random() < 0.7:
                cv2.line(img,pts[i],pts[j],(15,15,15),random.randint(1,3))
    bx1,by1 = max(0,min(p[0] for p in pts)-6), max(0,min(p[1] for p in pts)-6)
    bx2,by2 = min(w,max(p[0] for p in pts)+6), min(h,max(p[1] for p in pts)+6)
    labels.append(f"1 {((bx1+bx2)/2)/w:.6f} {((by1+by2)/2)/h:.6f} {(bx2-bx1)/w:.6f} {(by2-by1)/h:.6f}")

for split, count in [("train",NUM_TRAIN),("val",NUM_VAL)]:
    for idx in range(count):
        img = make_asphalt(IMG_SIZE, IMG_SIZE)
        labels = []
        if random.random() < 0.85:
            n = random.choices([1,2,3], weights=[0.6,0.3,0.1])[0]
            for _ in range(n):
                if random.random() < 0.6: draw_pothole(img, labels)
                else: draw_crack(img, labels)
        name = f"road_{split}_{idx:04d}"
        cv2.imwrite(str(DATASET_DIR/"images"/split/f"{name}.jpg"), img, [cv2.IMWRITE_JPEG_QUALITY,92])
        with open(DATASET_DIR/"labels"/split/f"{name}.txt","w") as f:
            f.write("\n".join(labels)+("\n" if labels else ""))

yaml_content = f"""path: /content/synthetic_road
train: images/train
val: images/val
names:
  0: Pothole
  1: Alligator Crack
  2: Longitudinal Crack
  3: Severe Edge Erosion
"""
DATA_YAML = "/content/synthetic_road/data.yaml"
with open(DATA_YAML, "w") as f:
    f.write(yaml_content)

print(f"✅ Synthetic dataset ready! Train: {NUM_TRAIN} | Val: {NUM_VAL}")
print(f"   data.yaml → {DATA_YAML}")

## 🔧 Step 4 — Set data.yaml Path

**Set `DATA_YAML` to wherever your dataset's `data.yaml` is located.**

In [ ]:
import os

# ── If you used Option A (Kaggle) ──────────────────────────────────────────
# Find the data.yaml automatically:
# import glob
# DATA_YAML = glob.glob("/content/dataset/**/*.yaml", recursive=True)[0]

# ── If you used Option B (Roboflow) ────────────────────────────────────────
# DATA_YAML is already set above ↑

# ── If you used Option C (Synthetic) ──────────────────────────────────────
DATA_YAML = "/content/synthetic_road/data.yaml"  # ← Change if using Kaggle/Roboflow

print(f"📁 Using dataset config: {DATA_YAML}")
assert os.path.exists(DATA_YAML), f"❌ data.yaml not found at {DATA_YAML}"

# Preview it
with open(DATA_YAML) as f:
    print("\n" + f.read())

## 🚀 Step 5 — Train YOLOv8 Model

In [ ]:
from ultralytics import YOLO
import torch

# ─────────────────────────────────────────────────
# ⚙️ TRAINING CONFIGURATION — Tune these:
# ─────────────────────────────────────────────────
BASE_MODEL  = "yolov8s.pt"   # yolov8n (fast) | yolov8s (balanced✅) | yolov8m (accurate) | yolov8l (best)
EPOCHS      = 50             # Recommended: 50–100 for real data, 30 for synthetic
BATCH_SIZE  = 16             # Reduce to 8 if you get CUDA out-of-memory
IMG_SIZE    = 640            # 640 (standard) or 1280 (higher accuracy, more VRAM)
PATIENCE    = 20             # Early stopping: stops if no improvement for 20 epochs
# ─────────────────────────────────────────────────

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️  Training on: {device.upper()}")
print(f"📦  Base model : {BASE_MODEL}")
print(f"📊  Dataset    : {DATA_YAML}")
print(f"🔁  Epochs     : {EPOCHS}")

model = YOLO(BASE_MODEL)

results = model.train(
    data        = DATA_YAML,
    epochs      = EPOCHS,
    imgsz       = IMG_SIZE,
    batch       = BATCH_SIZE,
    device      = device,
    project     = "/content/saferoad_runs",
    name        = "road_hazard_yolov8",
    patience    = PATIENCE,
    save        = True,
    # ─── Optimizer ───────────────────────────────
    optimizer   = "AdamW",
    lr0         = 0.001,
    lrf         = 0.01,
    # ─── Augmentations for road conditions ────────
    mosaic      = 1.0,     # Combines 4 images — great for small cracks
    mixup       = 0.15,    # Blends images — reduces false positives
    copy_paste  = 0.1,     # Copies objects between images
    hsv_h       = 0.015,   # Hue jitter — different asphalt types
    hsv_s       = 0.6,     # Saturation — wet vs dry road
    hsv_v       = 0.4,     # Brightness — shadows / tunnels
    degrees     = 10.0,    # Rotation — dashcam tilt
    translate   = 0.1,     # Translation — road bumps vibration
    scale       = 0.5,     # Scale — varying distance from road
    fliplr      = 0.5,     # Horizontal flip
    erasing     = 0.4,     # Random erasing — occlusion robustness
    verbose     = True
)

print("\n✅ Training complete!")

## 📊 Step 6 — Evaluate Model Performance

In [ ]:
# Run validation and print metrics
metrics = model.val()

print("\n" + "="*50)
print("  📊 SafeRoad AI — Final Model Metrics")
print("="*50)
print(f"  mAP@50      : {metrics.box.map50:.4f}   ({metrics.box.map50*100:.1f}%)")
print(f"  mAP@50-95   : {metrics.box.map:.4f}   ({metrics.box.map*100:.1f}%)")
print(f"  Precision   : {metrics.box.mp:.4f}")
print(f"  Recall      : {metrics.box.mr:.4f}")
print("="*50)

# Show training plots
from IPython.display import Image, display
import glob

run_dir = "/content/saferoad_runs/road_hazard_yolov8"
plots = glob.glob(f"{run_dir}/*.png")
for p in plots[:4]:  # Show up to 4 plots
    print(f"\n📈 {p.split('/')[-1]}")
    display(Image(p))

## 🧪 Step 7 — Test Inference on Sample Image

In [ ]:
import numpy as np
import cv2
import time
from IPython.display import Image, display

# Load best trained model
best_model_path = "/content/saferoad_runs/road_hazard_yolov8/weights/best.pt"
test_model = YOLO(best_model_path)

# ── Option A: Test on your own image ──────────────────
# from google.colab import files
# uploaded = files.upload()
# test_image_path = list(uploaded.keys())[0]

# ── Option B: Test on synthetic image (default) ───────
w, h = 640, 360
test_img = np.full((h,w,3), 85, dtype=np.uint8)
noise = np.random.normal(0, 12, (h,w,3)).astype(np.int16)
test_img = np.clip(test_img.astype(np.int16)+noise, 0, 255).astype(np.uint8)
cv2.line(test_img, (w//2,0), (w//2,h), (220,220,220), 8)
cv2.ellipse(test_img, (180,240), (60,35), 0, 0, 360, (20,20,20), -1)
cv2.ellipse(test_img, (180,240), (45,22), 0, 0, 360, (10,10,10), -1)
cv2.line(test_img, (400,100), (415,280), (18,18,18), 3)
test_image_path = "/content/test_road.jpg"
cv2.imwrite(test_image_path, test_img)

# Run inference
t0 = time.perf_counter()
results = test_model.predict(test_image_path, conf=0.25, verbose=False)
ms = (time.perf_counter()-t0)*1000

boxes = results[0].boxes
print(f"⚡ Inference: {ms:.1f} ms  |  {1000/max(ms,1):.0f} FPS")
print(f"🔍 Detections: {len(boxes)}")
for i,b in enumerate(boxes):
    cls_id = int(b.cls[0])
    name = test_model.names.get(cls_id, str(cls_id))
    conf = float(b.conf[0])*100
    print(f"  [{i+1}] {name} — {conf:.1f}% confidence")

# Save and show result
out_img = results[0].plot()
out_path = "/content/detection_result.jpg"
cv2.imwrite(out_path, out_img)
print("\n🖼️ Detection Result:")
display(Image(out_path))

## 💾 Step 8 — Download Trained Model

In [ ]:
import shutil
from google.colab import files

best_pt = "/content/saferoad_runs/road_hazard_yolov8/weights/best.pt"
last_pt = "/content/saferoad_runs/road_hazard_yolov8/weights/last.pt"

# Copy best.pt to /content for easy download
shutil.copy(best_pt, "/content/pothole_yolov8_best.pt")
print("✅ Model ready for download!")
print(f"   Size: {os.path.getsize(best_pt)/1e6:.2f} MB")

# Download it
files.download("/content/pothole_yolov8_best.pt")
print("📥 Download started! Save it as pothole_yolov8.pt in your server/models/ folder.")

## 📦 Step 9 — Export to ONNX (Optional — for web deployment)

In [ ]:
# Export to ONNX format for browser/mobile deployment
export_model = YOLO("/content/saferoad_runs/road_hazard_yolov8/weights/best.pt")
onnx_path = export_model.export(format="onnx", dynamic=True, simplify=True)
print(f"✅ ONNX model exported: {onnx_path}")

# Download ONNX
shutil.copy(onnx_path, "/content/pothole_yolov8.onnx")
files.download("/content/pothole_yolov8.onnx")
print("📥 ONNX model download started!")

---
## ✅ After Downloading

Place `pothole_yolov8_best.pt` in your project:
```
new road safe/
  server/
    models/
      pothole_yolov8.pt   ← rename and place here
```

Then start your detection server:
```bash
python server/ml_server.py
```

---
**🛣️ SafeRoad AI** — Making roads safer with AI